# 🏍️ Notebook 1: The Sidecar Pattern

A **sidecar** is a helper process that runs *right next to* your application — usually on the same host or in the same Kubernetes pod — and takes care of the *cross-cutting concerns* your app shouldn't have to reimplement: TLS, auth, logging, metrics, retries, service discovery.

### 🏍️ Analogy
A motorbike sidecar carries the groceries so the driver can focus on driving.
The bike = your app. The sidecar = the helper. They travel **together** and share the ride (the pod/host).

### Why this matters
This is the foundation of service meshes like **Envoy + Istio / Linkerd / Consul Connect**. Each pod gets a proxy sidecar that handles mTLS, routing, retries, and telemetry — without modifying any application code.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

No external services are needed — this notebook only uses the Python standard library.

## ❌ Bad: the monolithic handler does everything

When every service handles auth, logging, metrics, and retries inside its own code, you end up:
- re-implementing the same logic in every language your company uses,
- mixing business logic with infrastructure plumbing,
- having to redeploy the whole app just to change the log format.

In [ ]:
import time

# Pretend this is your HTTP handler. Each request goes through many concerns
# that have nothing to do with the business problem ("say hello").
def monolithic_handler(request):
    # 1) authentication
    if request.get('token') != 'secret':
        return {'status': 401, 'body': 'unauthorized'}
    # 2) logging
    print(f"[app-log] {request['method']} {request['path']}")
    # 3) metrics
    t0 = time.time()
    # 4) the *actual* business logic (this is the only part you'd want to write)
    body = {'hello': request.get('name', 'world')}
    print(f"[app-metric] duration={time.time()-t0:.4f}s")
    return {'status': 200, 'body': body}

print(monolithic_handler({'method':'GET','path':'/hi','token':'secret','name':'ada'}))
print(monolithic_handler({'method':'GET','path':'/hi','token':'bad','name':'ada'}))

## ⚠️ Better: extract cross-cutting concerns into a wrapper (in-process)

A first cleanup is to pull auth/log/metrics into a **wrapper** so the app contains only business logic. This is closer to middleware, not yet a true sidecar — but it's a useful stepping stone and makes the separation of responsibilities visible.

> 💡 A real sidecar runs as a **separate process**, not inside the app. We'll get there in Notebook 2.

In [ ]:
def business_handler(request):
    """Pure business code — no auth, no logging, no metrics."""
    return {'status': 200, 'body': {'hello': request.get('name', 'world')}}

class SidecarLike:
    """Intercepts every request and adds cross-cutting concerns.
    This is in-process middleware: same process as the app. It illustrates the
    *responsibility split* before we move to a real out-of-process sidecar.
    """
    def __init__(self, app):
        self.app = app

    def handle(self, request):
        # auth
        if request.get('token') != 'secret':
            return {'status': 401, 'body': 'unauthorized'}
        # logging + metrics around the call
        print(f"[sidecar-log] {request['method']} {request['path']}")
        t0 = time.time()
        resp = self.app(request)
        print(f"[sidecar-metric] duration={time.time()-t0:.4f}s")
        return resp

sc = SidecarLike(business_handler)
print(sc.handle({'method':'GET','path':'/hi','token':'secret','name':'ada'}))
print(sc.handle({'method':'GET','path':'/hi','token':'bad','name':'ada'}))

## 🧠 The key ideas of the sidecar pattern

| Property | What it means |
|---|---|
| **Co-located** | Runs on the same host / pod as the app, so calls between them are fast (localhost). |
| **Out-of-process** | It's a *separate* process — own binary, own memory, own lifecycle. |
| **Language-agnostic** | Because it talks to the app over the network (usually localhost HTTP/gRPC), the app can be in any language. |
| **Independently deployable** | You can ship a new sidecar (new TLS policy, new log format) without rebuilding the app image. |
| **Transparent** | The app doesn't know (or need to know) the sidecar exists; the sidecar intercepts traffic. |

### Classic sidecar jobs
- 🔒 mTLS termination / origination (encrypt pod-to-pod traffic)
- 🧭 service discovery + load balancing (Envoy, Linkerd-proxy)
- 🔁 retries, timeouts, circuit breaking
- 📈 metrics + distributed tracing (auto-instrument without code changes)
- 📜 log shipping (Fluent Bit, Vector)
- 🗝️ secret / config refresh (Vault agent)

### When *not* to use a sidecar
- Tiny single-language apps — a library might be cheaper than an extra process.
- Extremely latency-sensitive hot paths where the extra localhost hop matters.
- Resource-constrained environments (each sidecar uses CPU + memory per pod).

➡️ Next notebook: run the sidecar as a **real separate process** over a localhost HTTP hop.